# Qwen3-8B Inference — NL to CNL Translation

In [ ]:
from huggingface_hub import login
login('YOUR HUGGINGFACE_TOKEN')

In [ ]:
import torch
import json
import os
from pathlib import Path
from transformers import AutoTokenizer, pipeline
from peft import AutoPeftModelForCausalLM
from tqdm import tqdm
import pandas as pd
import requests
from datasets import load_dataset
import torch
from transformers import LogitsProcessor, PreTrainedTokenizer, AutoTokenizer, AutoModelForCausalLM
from lark import Lark, UnexpectedToken, UnexpectedCharacters, UnexpectedEOF
from abc import ABC, abstractmethod
from typing import Dict


os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("Libraries loaded.")
print(f"torch       : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
print(f"GPU         : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 0 · Qwen3 Output Cleaning Utility

In [ ]:
import re

def strip_think_tokens(text: str) -> str:
    """
    Removes Qwen3 chain-of-thought blocks from generated output.

    Qwen3 may prepend its answer with a <think>...</think> block when
    reasoning mode is active. This function strips that block (including
    empty ones like <think>\n\n</think>) and returns only the clean CNL text.

    Args:
        text: Raw string output from the model (may contain <think> block).

    Returns:
        Clean CNL string with all <think>...</think> content removed.
  
    """
    # Remove <think>...</think> blocks (including multiline, including empty ones)
    cleaned = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # Strip any leading/trailing whitespace or stray newlines left behind
    return cleaned.strip()



## 1 · Load Fine-Tuned Adapter

In [ ]:
# ── Update this path to your saved adapter / best checkpoint ──
adapter_path = "Path/To/Your/Saved/Adapter"

model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    device_map="auto",
    dtype=torch.bfloat16,        # training precision
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Model loaded from: {adapter_path}")
print(f"Device map: {model.hf_device_map}")

## 2 · Load Test Dataset

In [ ]:
## wrap your dataset loading and splitting

def load_data(path, test_size=0.1, seed=42):
  
    def create_conversation(sample):
        return {
            "messages": [
                {"role": "system",
                "content": "You are an expert in Translating the Natural language (NL) into Controlled Natural Language (CNL) translation. Always provide precise, syntactically correct translations of NL into CNL."},
                {"role": "user", "content": f"Translate the following natural language to controlled natural language: {sample['NL_V2']}"},
                {"role": "assistant", "content": sample["CNL_V2"]},
            ]
        }

    # Load and transform
    dataset = load_dataset("json", data_files=path, field="data_dict", split="train")
    dataset = dataset.map(create_conversation, remove_columns=dataset.features, batched=False)
    print("Dataset converted to conversational format.")

    # Split into train and test
    if test_size == 0:
        train_dataset = dataset
        return train_dataset, None
    split_dataset = dataset.train_test_split(test_size=test_size, seed=seed)
    train_dataset = split_dataset["train"]
    test_dataset = split_dataset["test"]

    train_dataset = train_dataset.shuffle(seed=seed)

    return train_dataset, test_dataset


In [ ]:
dataset_file = "Path/to/your/test_dataset.json" 
testset, _ = load_data(dataset_file, test_size=0)

# Optional: inspect an example
print("Example from test set:")
print(testset[0])

print(f"Train dataset size: {len(testset)}")
print(f"Test dataset size:  {len(testset)}")

## 3 · Decoder Class

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# Abstract base
# ─────────────────────────────────────────────────────────────────────────────

class CNLDecoder(ABC):    
    @abstractmethod
    def decode(self, prompt: str, max_new_tokens=100, temperature=0.5):
        """Method to make a sound"""
        pass
    

class CandidateGrammarLogitsProcessor(LogitsProcessor):
    
    def __init__(self, parser, partial_generation, tokenizer, candidate_limit=50):
        """
        valid_token_fn: A function that takes a token_id (int) and returns True/False.
        candidate_limit: How many top tokens to check. 50 is usually plenty.
        """
        self.tokenizer = tokenizer
        self.parser = parser
        self.candidate_limit = candidate_limit
        self.decoded_tokens_cache = self.build_decoded_tokens_cache(tokenizer)
        self.partial_generation = partial_generation 
        # Qwen3 special tokens used in the chat template
        self.prompt_tokens = ["<|im_start|>", "<|im_end|>", "<|endoftext|>"]  # Qwen3 chat template tokens
        
    @staticmethod
    def build_decoded_tokens_cache(tokenizer: PreTrainedTokenizer) -> Dict[int, str]:
        return {token_id: tokenizer.decode(token_id) for _, token_id in tokenizer.get_vocab().items()}
        
    def is_token_valid(self, text: str) -> bool:
        if not text:
            return False
        try:
            tree = self.parser.parse(text)
            return True
        except UnexpectedEOF as e:
            return True
        except UnexpectedCharacters as e:
            return False
        except UnexpectedToken as e:
            return True

    def __call__(self, input_ids, scores):
        batch_size = scores.shape[0]
        
        for i in range(batch_size):
            # Get the Top-K indices for this sequence
            top_k_values, top_k_indices = torch.topk(
                scores[i], 
                k=min(self.candidate_limit, scores.shape[-1]), 
                dim=-1
            )

            # Check validity for these candidates
            candidates = top_k_indices.tolist()
            valid_candidates = []
            
            checked_candidates = []
            for token_id in candidates:
                decoded_token = self.decoded_tokens_cache[token_id]
                if decoded_token in self.prompt_tokens or decoded_token.strip() == "be": #TODO Fix the grammar to properly deal with this problems "be".
                    valid_candidates.append(token_id)
                    continue
                
                copula_tokens = ["be","are","is","have","has"] #TODO FIx the grammar to properly deal with this problems. Maybe testing first the token and if it fail, test with space
                decoded_token = decoded_token if decoded_token.strip() not in copula_tokens else decoded_token +" "
                
                is_valid = self.is_token_valid(self.partial_generation + decoded_token)
                checked_candidates.append(f"-----{self.partial_generation }--: {decoded_token} ----- {is_valid}")
                if is_valid: 
                    valid_candidates.append(token_id)            
            print("\n".join(checked_candidates))

            if len(valid_candidates) > 0:
                new_scores = torch.full_like(scores[i], -float("inf"))
                
                for valid_id in valid_candidates:
                    new_scores[valid_id] = scores[i, valid_id]
                
                scores[i] = new_scores


        return scores


class ConstrainedCNLDecoder(CNLDecoder):
    
    def __init__(self, model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizer, parser: Lark, candidate_limit=50):
        self.tokenizer = tokenizer
        self.model = model
        self.candidate_limit = candidate_limit
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.parser = parser       
        
        
    def decode(self, prompt: str, max_new_tokens=512, temperature=0.1) -> str:
        
        if self.model is None:
            raise ValueError("Model is not loaded.")
        
        # --- Tokenize prompt ---
        encoded = self.tokenizer(prompt, return_tensors="pt")
        input_ids = encoded["input_ids"].to(self.device)
        attention_mask = encoded["attention_mask"].to(self.device)
        
        partial_generation = ""
        num_generated = 0
        
        print(num_generated)
        
        # --- Normalize EOS / PAD token IDs ONCE ---
        stop_ids = self.model.config.eos_token_id
        if isinstance(stop_ids, int):
            stop_ids = {stop_ids}
        else:
            stop_ids = set(stop_ids)
        
        while num_generated < max_new_tokens:
            
            grammar_processor = CandidateGrammarLogitsProcessor(
                partial_generation=partial_generation,
                tokenizer=self.tokenizer,
                parser=self.parser,
                candidate_limit=self.candidate_limit
            )
            
            output = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=1,
                do_sample=False, ##True
                temperature=temperature,
                logits_processor=[grammar_processor],
            )

            # Extract only the new token
            new_token_id = output[0, -1].unsqueeze(0).unsqueeze(0)
            
            gen_token = self.tokenizer.decode(new_token_id[0], skip_special_tokens=False)
            
            # Stop on EOS
            if new_token_id.item() in stop_ids:
                break
            
            partial_generation += gen_token
                    
            # Append it manually
            input_ids = torch.cat([input_ids, new_token_id], dim=1)
            attention_mask = torch.cat(
                [attention_mask, torch.ones_like(new_token_id)], dim=1
            )
            num_generated += 1
            
        # Remove Qwen3 <think>...</think> block from constrained output
        partial_generation = strip_think_tokens(partial_generation)
        return partial_generation
    
class NaiveCNLDecoder(CNLDecoder):
    
    def __init__(self, model: AutoModelForCausalLM, tokenizer: PreTrainedTokenizer):
        self.tokenizer = tokenizer
        self.model = model
        self.device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
        self.pipeline = pipeline(
            "text-generation",
            model=self.model,
            tokenizer=self.tokenizer,
            device_map=self.device
        )
        
        
    def decode(self, prompt: str, max_new_tokens=512, temperature=0) -> str:
        outputs = self.pipeline(
                prompt,
                max_new_tokens=max_new_tokens,
                do_sample=False,          # greedy decoding
                temperature=temperature,
                top_p=None,
                eos_token_id=self.tokenizer.eos_token_id,
                pad_token_id=self.tokenizer.pad_token_id,
            )
        # Extract generated text
        generated_text = outputs[0]['generated_text']
        prediction = generated_text[len(prompt):].strip()
        
        # Clean up potential extra eos tokens
        if self.tokenizer.eos_token in prediction:
            prediction = prediction.split(self.tokenizer.eos_token)[0].strip()
        
        # Remove Qwen3 <think>...</think> chain-of-thought block if present
        prediction = strip_think_tokens(prediction)
        
        return prediction

## 4 · SyntaxTester (with checkpoint saving)

In [ ]:

SYNTAX_URL = "http://your-server/api/check_syntax" # Replace with your server IP and compile API endpoint
COMPILE_URL = "http://your-server/api/compile"  # Replace with your server IP and compile API endpoint
API_KEY     = "API_KEY_HERE"   # Replace with your actual API key
HEADERS     = {"Content-Type": "application/json", "X-API-KEY": API_KEY}

class SyntaxTester:
    def __init__(self, decoder: CNLDecoder, tokenizer: PreTrainedTokenizer, test_dataset):
        self.test_dataset = test_dataset
        self.decoder = decoder
        self.tokenizer = tokenizer
        self.error_count = 0
        self.wrong_indexes = []
        self.predictions = []

    def evaluate(self, verbose=False):
        for idx, sample in enumerate(self.test_dataset):
            input_nl = sample["messages"][1]["content"].replace(
                "Translate the following natural language to controlled natural language: ", ""
            ).strip()
            predicted_cnl = self.__predict(input_nl)
            if verbose:
                print(f"Processing Sample {idx+1} of {len(self.test_dataset)}")

            if not self.__syntax_check(predicted_cnl):
                self.error_count += 1
                self.wrong_indexes.append(idx)
            self.predictions.append({
                "input_nl": input_nl,
                "predicted_cnl": predicted_cnl
            })
        total_samples = len(self.test_dataset)
        accuracy = (total_samples - self.error_count) / total_samples if total_samples > 0 else 0.0

        if verbose:
            print(f"Total samples: {total_samples}")
            print(f"Syntax errors: {self.error_count}")
            print(f"Accuracy (no syntax errors): {accuracy:.4f}")

        return accuracy


    def __predict(self, input_text, max_new_tokens=512):        

        prompt = self.tokenizer.apply_chat_template(
            [
                {"role": "system",
                 "content": "You are an expert in Translating the Natural language (NL) into Controlled Natural Language (CNL) translation. Always provide precise, syntactically correct translations of NL into CNL."},
                {"role": "user", "content": f"Translate the following natural language to controlled natural language: {input_text} "}
            ],
            tokenize=False,
            add_generation_prompt=True
        )

        return self.decoder.decode(prompt, max_new_tokens=max_new_tokens, temperature=0.1)
    

    def __syntax_check(self, cnl_text):
        url = "http://your-server/api/check_syntax"  # Replace with your actual endpoint 240 to 29

        payload = {
            "cnls": cnl_text
        }
        headers = {
            'Content-Type': 'application/json',
            "X-API-KEY": "API_KEY_HERE"
        }

        try:
            response = requests.post(url, headers=headers, data=json.dumps(payload))
            response.raise_for_status()  # Raise an error for bad status codes
            result = response.json()
            if(result is None or "cli_message" not in result or result["cli_message"] != "Input file fits the grammar."):
                return False
            return True
        except:
            return False

print("yntaxTester defined.")

## 5 · Decoder & Run a Single NL Test

In [ ]:

single_NL = "INPUT_NL_HERE"  # Replace with a single natural language sentence for testing

#-------NORMAL DECODING----------------
decoder: CNLDecoder = NaiveCNLDecoder(model=model, tokenizer=tokenizer)



tester = SyntaxTester(decoder, tokenizer=tokenizer,  test_dataset=[{"messages": [
    {"role": "system",
     "content": "You are an expert in Translating the Natural language (NL) into Controlled Natural Language (CNL) translation. Always provide precise, syntactically correct translations of NL into CNL."},
    {"role": "user", "content": f"Translate the following natural language to controlled natural language, and be sure to analyse properly the semantics of NL. DO NOT use vertex word in the CNL text: {single_NL} "}
]}])  
results = tester.evaluate(verbose=True)  
print(results)      
wrong_sentences = [tester.predictions[i] for i in tester.wrong_indexes]
print(tester.predictions)
wrong_sentences

### 6 · ASP Generator

In [ ]:


class ASPGenerator(SyntaxTester):
    def __init__(self, decoder, tokenizer, test_dataset):
        # Initializing the parent SyntaxTester
        super().__init__(decoder, tokenizer, test_dataset)
        self.data_dict = []
        self.compilation_errors = 0

    def __get_syntax_status(self, cnl_text):
        """Internal helper to check grammar syntax via API."""
        url = "http://your-server/api/check_syntax"
        payload = {"cnls": cnl_text}
        headers = {
            'Content-Type': 'application/json',
            "X-API-KEY": "API_KEY_HERE"
        }
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=10)
            result = response.json()
            # Returns True if the grammar matches, False otherwise
            return result.get("cli_message") == "Input file fits the grammar."
        except Exception:
            return False

    def __get_asp(self, cnl_text):
        """Internal helper to compile CNL into ASP via API."""
        url = "http://your-server/api/compile"
        payload = {"cnls": cnl_text}
        headers = {
            'Content-Type': 'application/json',
            "X-API-KEY": "API_KEY_HERE"
        }
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=15)
            response.raise_for_status()
            result = response.json()
            asp_content = result.get("asp", "")
            
            if asp_content and len(asp_content.strip()) > 0:
                return asp_content
            else:
                self.compilation_errors += 1
                return "ERROR! NOT COMPILING"
        except Exception:
            self.compilation_errors += 1
            return "ERROR! NOT COMPILING"

    def process_and_save(self, json_path, output_filename="file_name.csv", verbose=True):
        # 1. Load the source JSON directly to preserve all metadata (ASP, Category, etc.)
        with open(json_path, 'r') as f:
            raw_data = json.load(f)
        
        samples = raw_data.get("data_dict", [])
        self.data_dict = []
        self.compilation_errors = 0
        total_samples = len(samples)
        
        print(f"Starting Qwen3-8B Pipeline: Processing {total_samples} samples...")

        for item in tqdm(samples, disable=not verbose):
            # Extract raw data from JSON structure
            nl_input = item.get('NL_V2', '')
            actual_cnl = item.get('CNL_V2', '')
            actual_asp = item.get('ASP', '')
            category = item.get('Category', 'N/A')
            id = item.get('ID', 'N/A')

            # 2. Generate Prediction (using the decoder from parent class)
            predicted_cnl = self._SyntaxTester__predict(nl_input)

            # 3. Check Syntax (API)
            is_valid_syntax = self.__get_syntax_status(predicted_cnl)

            # 4. Generate ASP (API)
            generated_asp = self.__get_asp(predicted_cnl)

            # 5. Format results exactly like the T5-small output
            self.data_dict.append({
                'Natural Language': nl_input,
                'Actual CNL': actual_cnl,
                'Predicted CNL': predicted_cnl,
                'Syntax Valid': is_valid_syntax,
                'Generated ASP': generated_asp,
                'Actual ASP': actual_asp,
                'Category': category,
                'ID': id
            })

        # --- Save Only to CSV ---
        results_df = pd.DataFrame(self.data_dict)
        results_df.to_csv(output_filename, index=False)

        if verbose:
            syntax_acc = (results_df['Syntax Valid'].sum() / total_samples) * 100
            print("\n" + "="*40)
            print("QWEN3 EVALUATION SUMMARY")
            print("="*40)
            print(f"Total Samples:      {total_samples}")
            print(f"Syntax Accuracy:    {syntax_acc:.2f}%")
            print(f"ASP Success Rate:   {((total_samples - self.compilation_errors)/total_samples)*100:.2f}%")
            print(f"CSV saved to:       {output_filename}")
            print("="*40)

        return self.data_dict

print("ASPGenerator defined.")

## 7 · Run Full Evaluation Pipeline

In [ ]:
# ── Set your test JSON file path ──
dataset_file = "path/to/your/test_dataset.json"

#-------NORMAL DECODING (default)----------------
decoder: CNLDecoder = NaiveCNLDecoder(model=model, tokenizer=tokenizer)


# Initialize generator with the full test dataset
generator = ASPGenerator(decoder=decoder, tokenizer=tokenizer, test_dataset=None)

# Run full pipeline — saves CSV automatically
results = generator.process_and_save(
    json_path=dataset_file,
    output_filename="path/to/your/outputfilename.csv"
)